# packages


In [1]:
# data treatment and manipulation
import pandas as pd # treatment of dataframes
import numpy as np # scalar and vector operations
from sklearn.preprocessing import StandardScaler

# geometric data manipulation
import geopandas as gp # it is used to manipulate geometric data, it is an extension of pandas
from shapely.geometry import Point, MultiPolygon, GeometryCollection
from shapely.ops import unary_union
from shapely import wkt

# packages for TDA
import gudhi # pesistence homology, complexes, persistence diagrams, etc
from scipy.spatial import Delaunay # alpha complex
from ripser import ripser # ripser is a package for computing persistent homology, it is faster than gudhi for large datasets
from gudhi.wasserstein import wasserstein_distance # wasserstein distance between persistence diagrams

# gudhi complments
import ripser
import persim

# drawings
import matplotlib.pyplot as plt # 

# time complexity
import time


# Data 

In [2]:
df = pd.read_csv('/home/niko/semillero_topology_data_analysis/Centroides_por_departamento.csv') # read the csv modified in cell before

In [3]:
df['X'] = df['geometry'].apply(lambda x: wkt.loads(x).centroid.x if isinstance(x, str) and x.startswith('POINT') else np.nan)
df['Y'] = df['geometry'].apply(lambda x: wkt.loads(x).centroid.y if isinstance(x, str) and x.startswith('POINT') else np.nan)

In [4]:
gdf = gp.GeoDataFrame(df, geometry=gp.points_from_xy(df['X'], df['Y']), crs="EPSG:4326")

In [5]:
lista_departamentos = np.array(['Caquetá', 'Amazonas', 'Putumayo', 'Nariño',
       'Cauca', 'Guaviare', 'Huila', 'Meta',
       'Bolívar', 'Valle del Cauca', 'Tolima',
       'Cundinamarca', 'Vichada', 'Chocó', 'Casanare',
       'Quindío', 'Risaralda', 'Boyacá', 'Caldas',
       'Antioquia', 'Arauca', 'Santander',
       'Norte de Santander', 'Córdoba', 'Cesar', 'Sucre',
       'Magdalena', 'Atlántico', 'La Guajira']) # uso esto para extraer todos los nombres que quiero en un array.

datos_filtrados = {} # diccionario vacio en el que quiero todos los geo data frames con llaves nombres de departamentos

for dep in lista_departamentos: # para cada nombre en la lista que no importa que sea un array, es como for i in range(0, len(lista_departamentos))
    filtro = gdf['DeNombre'].str.contains(dep, case = False, na = False) # filtramos que el litro sea el data frame geometrico en el nombre y que contenga algo de la palabra de la lista
    datos_filtrados[dep] = gdf[filtro].copy() # datos filtrados[nombre del departamento como lo tenemos en la lista] es igual a la copia del geometrico en el nombre que escogimos
    # esto es importante para acceder a los valores geometricos, accedemos mediante gdf no mediante datos_filtrados, ya que es clave = valor


In [6]:
for nombre, gdf_depto in datos_filtrados.items(): # hacemos un for doble en el diccionario, items por llaves
    datos_filtrados[nombre]['x'] = gdf_depto.geometry.x
    datos_filtrados[nombre]['y'] = gdf_depto.geometry.y

# data Scaling (standard scaler)

In [7]:
 # libreria sklearn.preprocessing, es un tipo de escalamiento de datos usando desviación estandar y promedio pero de todas las coordenadas no cada una

todas = np.vstack([
    np.column_stack((gdf_depto['x'], gdf_depto['y']))
    for gdf_depto in datos_filtrados.values()
])

# ================================
# 2. Ajustar scaler UNA sola vez
# ================================
scaler = StandardScaler()
scaler.fit(todas)

# ================================
# 3. Transformar cada departamento
# ================================
for nombre, gdf_depto in datos_filtrados.items():

    # Obtener coords originales
    coords = np.column_stack((gdf_depto['x'], gdf_depto['y']))

    # Transformar usando el MISMO scaler global
    coords_scaled = scaler.transform(coords)

    # Guardar las coords escaladas en el GeoDataFrame
    datos_filtrados[nombre][['x', 'y']] = coords_scaled

    print(f"{nombre}: escalado OK")


Caquetá: escalado OK
Amazonas: escalado OK
Putumayo: escalado OK
Nariño: escalado OK
Cauca: escalado OK
Guaviare: escalado OK
Huila: escalado OK
Meta: escalado OK
Bolívar: escalado OK
Valle del Cauca: escalado OK
Tolima: escalado OK
Cundinamarca: escalado OK
Vichada: escalado OK
Chocó: escalado OK
Casanare: escalado OK
Quindío: escalado OK
Risaralda: escalado OK
Boyacá: escalado OK
Caldas: escalado OK
Antioquia: escalado OK
Arauca: escalado OK
Santander: escalado OK
Norte de Santander: escalado OK
Córdoba: escalado OK
Cesar: escalado OK
Sucre: escalado OK
Magdalena: escalado OK
Atlántico: escalado OK
La Guajira: escalado OK


In [8]:
t0 = time.time() # inicio temporizador
diagramas_persistencia = {}
for nombre, gdf in datos_filtrados.items():
    # misma estructura para recorrer diccionario sin errores
    coordenadas = np.column_stack((gdf.geometry.x, gdf.geometry.y)) # almacenar coordenadas en columna ya que solo son numeros, lo almacenamos en una matriz de 2 columnas por
    # n como tamaño del departamento
    alpha_complex = gudhi.AlphaComplex(points=coordenadas) # complejo alpha de gudhi usando ya (x,y) complejo alpha 2 dimensiones
    simplex_tree = alpha_complex.create_simplex_tree() # complejo simplicial
    diag = simplex_tree.persistence() # el diagrama

    # save in a dictionary
     # diccionario para guardar los diagramas de persistencia
    diagramas_persistencia[nombre] = diag
    
t1 = time.time()  # fin temporizador

print(f"Tiempo total: {t1 - t0:.6f} segundos")   # imprimir tiempo

Tiempo total: 0.075391 segundos


In [ ]:
for nombre, gdf in datos_filtrados.items():
    # misma estructura para recorrer diccionario sin errores
    coordenadas = np.column_stack((gdf.geometry.x, gdf.geometry.y)) # almacenar coordenadas en columna ya que solo son numeros, lo almacenamos en una matriz de 2 columnas por
    # n como tamaño del departamento
    alpha_complex = gudhi.AlphaComplex(points=coordenadas) # complejo alpha de gudhi usando ya (x,y) complejo alpha 2 dimensiones
    simplex_tree = alpha_complex.create_simplex_tree() # complejo simplicial
    diag = simplex_tree.persistence() # el diagrama

    # save in a folder 
     # diccionario para guardar los diagramas de persistencia
    # save in this path /home/niko/semillero_topology_data_analysis/images
    plt.savefig(f"/home/niko/semillero_topology_data_analysis/images/{nombre}_persistence_diagram.png")

<Figure size 640x480 with 0 Axes>